# [서연] TOPIS 월별 교통량 수집·전처리 실행 기록

실제 기능은 crawling/traffic_volume.py에 두고, 이 노트북에서는 코드 버전·과거 성공/실패/중단 기록·오류·재시도·화면 캡처·결과 표를 확인한다.

기록 조회 셀은 Selenium을 실행하지 않는다. 1개월 시험은 EXECUTE_ONE_MONTH_TEST, 전체 수집은 EXECUTE_COLLECTION 값을 직접 True로 바꾼 경우에만 시작한다.

## [서연] 1. 커널·패키지·프로젝트 경로 확인

In [ ]:
# [서연] 환경 확인 — Selenium·네트워크·파일 저장 없음
import importlib.util
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'README.md').is_file() and (candidate / 'crawling' / 'traffic_volume.py').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
packages = ['selenium', 'pandas', 'openpyxl', 'pyarrow']
package_status = {name: importlib.util.find_spec(name) is not None for name in packages}
print('Python:', sys.version)
print('실행 파일:', sys.executable)
print('프로젝트:', ROOT)
print('패키지:', package_status)

## [서연] 2. 핵심 모듈과 코드 버전 확인

In [ ]:
# [서연] 핵심 함수 코드와 Git 상태 표시 — 수집 실행 없음
import inspect
from crawling import traffic_volume as TV

provenance_now = TV.git_provenance()
display(provenance_now)
for function in [TV.required_months, TV.download_required_months, TV.preprocess_downloads, TV.run_pipeline]:
    print('\n' + '=' * 90)
    print(function.__name__)
    print('=' * 90)
    print(inspect.getsource(function))

## [서연] 3. 정희의 90일과 필요한 26개월 확인

In [ ]:
# [서연] 대상 기간 확인 — 수집 실행·파일 저장 없음
target_days = TV.load_target_days()
target_months = TV.required_months(target_days)
display(target_days.groupby(['split', 'role']).size().rename('days').reset_index())
print('날짜 범위:', target_days.date.min().date(), '~', target_days.date.max().date())
print('필요 연월:', len(target_months), '개')
print(target_months)

## [서연] 4. 과거 실행 이력 전체 조회
성공 기록만 고르지 않고 성공·실패·중단 폴더를 모두 읽는다. 이 셀을 다시 실행해도 기존 기록을 변경하지 않는다.

In [ ]:
# [서연] 과거 실행 요약 — Selenium·네트워크·파일 저장 없음
history = TV.list_run_history()
display(history if not history.empty else '아직 실행 기록이 없습니다.')

## [서연] 5. 오류·재시도 로그 전체 조회

In [ ]:
# [서연] 모든 실행의 JSONL 로그 표시 — 수집 실행 없음
import json
import pandas as pd

run_roots = [TV.data_root() / TV.RUN_SUBDIR, TV.data_root() / TV.LEGACY_RUN_SUBDIR]
log_rows = []
for run_root in run_roots:
    if not run_root.is_dir():
        continue
    for path in sorted(run_root.glob('**/yang_seoyeon_collection_log_*.jsonl')):
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                row = {'status': 'log_parse_error', 'error': str(exc)}
            row['run_folder'] = path.parent.name
            row['line_number'] = line_number
            log_rows.append(row)
log_history = pd.DataFrame(log_rows)
display(log_history if not log_history.empty else '아직 수집 로그가 없습니다.')
if not log_history.empty:
    display(log_history.groupby(['run_folder', 'status']).size().rename('count').reset_index())

## [서연] 6. 과거 Selenium 화면 캡처 확인

In [ ]:
# [서연] 성공·실패·중단 실행의 캡처 표시 — 수집 실행 없음
from IPython.display import Image, display

screenshots = sorted({path.resolve() for root in run_roots if root.is_dir() for path in root.glob('**/screenshots/yang_seoyeon_*.png')})
print('화면 캡처:', len(screenshots), '개')
for path in screenshots:
    print(path.parent.parent.name, '/', path.name)
    display(Image(filename=str(path), width=900))

## [서연] 7. 1개월 테스트 명시적 실행
전체 수집 전에 필요한 26개월 중 한 달만 시험한다. EXECUTE_ONE_MONTH_TEST를 직접 True로 바꾼 경우에만 Selenium이 실행된다. 성공·실패·중단·재시도·스크린샷은 전체 실행과 같은 방식으로 별도 보존된다.

In [ ]:
# [서연] 1개월 Selenium 테스트 — 기본값 False이므로 자동 실행되지 않음
EXECUTE_ONE_MONTH_TEST = False
TEST_MONTH = '2024-11'

if not EXECUTE_ONE_MONTH_TEST:
    print('1개월 테스트를 실행하지 않았습니다. 실행할 때만 True로 변경하세요.')
else:
    one_month_test_result = TV.run_test_month(
        TEST_MONTH,
        headless=False,
        retries=3,
        wait_seconds=60,
        progress=print,
    )
    display(one_month_test_result)

## [서연] 8. 전체 26개월 수집·전처리 명시적 실행
이 셀만 Selenium, 네트워크, XLSX 다운로드, Parquet 저장을 발생시킨다. 실행하려면 EXECUTE_COLLECTION을 직접 True로 바꾼다. 전용 Chrome이 닫힐 때까지 창을 조작하지 않는다.

In [ ]:
# [서연] 실제 수집 실행 셀 — 기본값 False이므로 자동 실행되지 않음
EXECUTE_COLLECTION = False

if not EXECUTE_COLLECTION:
    print('수집하지 않았습니다. 직접 실행할 때만 EXECUTE_COLLECTION = True로 변경하세요.')
else:
    run_result = TV.run_pipeline(
        headless=False,
        retries=3,
        wait_seconds=60,
        progress=print,
    )
    display(run_result)

## [서연] 9. 실행 후 이력·검증·결과 표 새로 읽기
실제 수집이 끝난 뒤 이 셀을 실행하고 노트북을 저장한다. 이전 실패 기록까지 포함해 교수님이 확인할 수 있다.

In [ ]:
# [서연] 실행 후 결과 요약 — Selenium·네트워크·파일 저장 없음
history_after = TV.list_run_history()
display(history_after if not history_after.empty else '실행 기록이 없습니다.')

if not history_after.empty:
    for run_dir_text in history_after.run_dir:
        run_dir = Path(run_dir_text)
        for validation_path in sorted(run_dir.glob('yang_seoyeon_*validation*.csv')):
            print(run_dir.name, '/', validation_path.name)
            display(pd.read_csv(validation_path, encoding='utf-8-sig'))
        for sample_path in sorted(run_dir.glob('yang_seoyeon_test_sample_*.csv')):
            print(run_dir.name, '/', sample_path.name)
            display(pd.read_csv(sample_path, encoding='utf-8-sig').head(20))
        for summary_path in sorted(run_dir.glob('yang_seoyeon_summary_*.json')):
            print(run_dir.name, '/', summary_path.name)
            display(json.loads(summary_path.read_text(encoding='utf-8')))

## [서연] 10. 수집 데이터와 최종 결과 소량 확인

In [ ]:
# [서연] 원본 목록과 최종 Parquet 집계·미리보기 — 읽기 전용
raw_files = sorted((TV.data_root() / TV.RAW_SUBDIR).glob('yang_seoyeon_topis_traffic_volume_*.xlsx'))
raw_table = pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size, 'modified_at': pd.Timestamp(p.stat().st_mtime, unit='s')} for p in raw_files])
display(raw_table)

processed_files = sorted((TV.data_root() / TV.PROCESSED_SUBDIR).glob('yang_seoyeon_traffic_volume_*_(성공).parquet'))
for path in processed_files:
    frame = pd.read_parquet(path)
    print(path.name, '행:', len(frame), '날짜:', frame.date.nunique(), '지점:', frame.spot_id.nunique())
    display(frame.head(10))
    display(pd.DataFrame([{
        'traffic_volume_missing': int(frame.traffic_volume.isna().sum()),
        'coordinate_missing': int(frame[['latitude', 'longitude']].isna().any(axis=1).sum()),
        'duplicate_keys': int(frame.duplicated(['date', 'spot_id', 'io_type', 'hour']).sum()),
    }]))

## [서연] 11. 제출 전 저장
1~6번 조회 셀과 9~10번 결과 셀을 실행한 뒤 Jupyter의 Save 기능으로 노트북을 저장한다. 실행 번호·오류·표·스크린샷 출력이 노트북에 남는다. 실제 수집 셀은 다시 실행하지 않아도 과거 실행 폴더를 통해 기록을 계속 확인할 수 있다.